In [1]:
### import torch
from pair_contrast.pair_contrast_model import LlamaForContrastive, BartForContrastive
from pair_contrast.pair_contrast_data import DataCollatorForContrastive,ContrastiveDataset
from transformers import AutoTokenizer
from pair_contrast.pair_contrast_trainer import ContrastiveTrainer, LlamaContrastiveTrainer
import numpy as np
import torch


# model base
model_base = "meta-llama/Llama-3.2-1B-Instruct"

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_base)
tokenizer.pad_token = tokenizer.eos_token
# load model
model = LlamaForContrastive.from_pretrained(model_base)

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# contoh input (pairwise)
dialog = "Budi:Hi andi!\nAndi:Hi bud :v"
summary = "andi dan budi saling sapa"
full_prompt = "Ringkas dialog ini ##Dialog:\n" + dialog + "\n##Summary\n" + summary
instruct_only = "Ringkas dialog ini ##Dialog:\n" + dialog + "\n##Summary\n"

# tokenize
inputs = tokenizer(
    full_prompt,
    return_tensors="pt",
    max_length=32,
    padding="max_length",
    truncation=True
)

instruct = tokenizer(
    instruct_only,
    return_tensors="pt",
    truncation=True
)

label_ids = inputs['input_ids'].clone().detach()
prompt_len = len(instruct["input_ids"][0])
                
# Pastikan tidak melebihi panjang total jika terjadi truncation
mask_limit = min(prompt_len, len(inputs['input_ids'][0]))
label_ids[:,:mask_limit] = -100

# pindah ke device
inputs = {k: v.to(device) for k, v in inputs.items()}
# inputs['labels'] = label_ids

# inference
# with torch.no_grad():
#     outputs = model(**inputs)

# biasanya output berupa score/logits

/home/qiscus/Coresearch_ITB_Qiscus/repo_alan/SLM Summarization/symbolic_distill_contrastive_summ/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
LlamaForContrastive has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin

In [2]:
collator = DataCollatorForContrastive(
    tokenizer=tokenizer,
    model=model,
    is_decoder_only=True
)

data_dir = "data/dialogue_train.json"

train_dataset = ContrastiveDataset(data_dir, tokenizer)

/home/qiscus/Coresearch_ITB_Qiscus/repo_alan/SLM Summarization/symbolic_distill_contrastive_summ/venv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:4117: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [3]:
len(train_dataset)

6

In [4]:
import torch
import gc

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

cleanup()

In [5]:
from transformers import TrainingArguments, Seq2SeqTrainingArguments

training_arguments = TrainingArguments(
    output_dir="outputs",
    remove_unused_columns=False,
    
    # 1. Perkecil batch size dan gunakan Gradient Accumulation
    per_device_train_batch_size=1, 
    gradient_accumulation_steps=4, # Total batch size efektif = 1 * 4 = 4    
    # 2. Aktifkan Campuran Presisi (Mixed Precision)
    # Gunakan bf16=True jika kartu grafis kamu seri RTX 3000/4000 (Ampere/Lovelace)
    # Jika seri lama, gunakan fp16=True
    bf16=True, 
    
    # 3. Gradient Checkpointing (Paling Ampuh hemat VRAM, tapi sedikit lebih lambat)
    gradient_checkpointing=True,
    
    # 4. Gunakan Optimizer yang lebih efisien (8-bit)
    # Membutuhkan library 'bitsandbytes'
    optim="paged_adamw_8bit",
    
    # Tambahan: Batasi log/evaluasi agar tidak menumpuk di memori
    logging_steps=10,
)

trainer = LlamaContrastiveTrainer(
            model=model,
            args=training_arguments,
            theta=1.0,
            train_dataset=train_dataset,
            data_collator=collator,
            tokenizer=tokenizer,
        )

trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss


TrainOutput(global_step=3, training_loss=10.099979400634766, metrics={'train_runtime': 28.6993, 'train_samples_per_second': 0.627, 'train_steps_per_second': 0.105, 'total_flos': 271218866503680.0, 'train_loss': 10.099979400634766, 'epoch': 2.0})

In [6]:
print(inputs['labels'])

KeyError: 'labels'

In [ ]:
print(inputs['input_ids'])